# BinSense — M2b: Advanced EDA + Image Analysis

Closes the gaps vs the evaluator **Week 2&3** template that `02_eda_and_splits`
didn't cover:
- **Advanced EDA** (Set 3): weight box/violin/KDE, dimension **correlation heatmap +
  Spearman**, bin-level weight / overloaded-bin analysis, single-vs-multi pie.
- **Image analysis** (Set 4): **SNR, entropy, edge density, Sobel energy, contour
  analysis** — plus a multi-criterion poor-quality flag.

Heavy image math lives in `tools/eda/image_metrics.py`; this notebook is the
narrative that calls it. Reads the master tables produced by `02_eda_and_splits`.

In [ ]:
# Cell 1: Bootstrap — path resolution + data/code split
import sys, os, subprocess
from pathlib import Path

GITHUB_URL = 'https://github.com/rishib09/AmazonBinSense.git'
DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'
LOCAL_DATA = r'G:\My Drive\Interview Kickstart\Capstone Project\Amazon BinSense\data'

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/AmazonBinSense')
    if (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(PROJECT_ROOT)], check=True)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(DRIVE_ROOT) / 'data')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'opencv-python-headless', 'seaborn', 'tqdm'], check=True)
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    if not os.getenv('BINSENSE_DATA_DIR') and Path(LOCAL_DATA).exists():
        os.environ['BINSENSE_DATA_DIR'] = LOCAL_DATA

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Running in:', 'Google Colab' if IN_COLAB else 'Local', '| ROOT:', PROJECT_ROOT)

In [ ]:
# Cell 2: Imports + load the master tables from 02_eda_and_splits
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from utils.env_utils import setup_env

cfg = setup_env(verbose=True)
sns.set_theme(style='whitegrid')

bins = pd.read_csv(cfg.splits_dir / 'bins_master.csv')
items = pd.read_csv(cfg.splits_dir / 'items_master.csv')
print('bins:', bins.shape, '| items:', items.shape)

## 1. Weight distribution — box / violin / KDE
(Template Set 3.2 — we previously only did histograms.)

In [ ]:
w = items['weight'].dropna(); w = w[w > 0]
logw = np.log1p(w)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(x=w, ax=ax[0], color='darkorange');   ax[0].set(title='Weight — box', xlabel='lbs')
sns.violinplot(x=logw, ax=ax[1], color='purple');  ax[1].set(title='log(1+weight) — violin', xlabel='log lbs')
sns.kdeplot(logw, fill=True, ax=ax[2], color='teal'); ax[2].set(title='log(1+weight) — KDE', xlabel='log lbs')
plt.tight_layout(); plt.show()
print(w.describe().round(2).to_string())

## 2. Dimension correlation — Pearson + Spearman heatmaps
(Template Set 3.6 — we previously reported only a single weight-vs-volume r.)

In [ ]:
num = items[['weight', 'volume', 'length', 'width', 'height']].dropna()
num = num[(num > 0).all(axis=1)]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(num.corr('pearson'),  annot=True, cmap='coolwarm', center=0, ax=ax[0]); ax[0].set_title('Pearson')
sns.heatmap(num.corr('spearman'), annot=True, cmap='coolwarm', center=0, ax=ax[1]); ax[1].set_title('Spearman')
plt.tight_layout(); plt.show()

## 3. Bin-level weight & load — spotting overloaded bins
(Template Set 3.4 — total weight per bin and its relationship to item count.)

In [ ]:
it = items.copy()
it['line_weight'] = it['weight'] * it['quantity']
binw = (it.groupby('bin_id')
          .agg(total_weight=('line_weight', 'sum'),
               n_units=('quantity', 'sum'),
               n_asins=('asin', 'nunique'))
          .reset_index())
print('Heaviest bins:')
display(binw.sort_values('total_weight', ascending=False).head(10))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].scatter(binw['n_units'], binw['total_weight'], s=6, alpha=0.3)
ax[0].set(title='total weight vs #units', xlabel='#units', ylabel='lbs')
ax[1].scatter(binw['n_asins'], binw['total_weight'], s=6, alpha=0.3)
ax[1].set(title='total weight vs #distinct ASINs', xlabel='#ASINs', ylabel='lbs')
plt.tight_layout(); plt.show()
print(f"corr(#units, total_weight) = {binw['n_units'].corr(binw['total_weight']):.2f}")

## 4. Single vs multi-item composition
(Template Set 3.7 — pie view of the architecture-driving split.)

In [ ]:
single = int((bins['asin_count'] == 1).sum())
multi   = int((bins['asin_count'] >= 2).sum())
bucket  = bins['bucket'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].pie([single, multi], labels=['single-ASIN', 'multi-ASIN'], autopct='%1.1f%%',
          colors=['steelblue', 'indianred']); ax[0].set_title('Single vs multi')
ax[1].pie(bucket.values, labels=bucket.index, autopct='%1.1f%%'); ax[1].set_title('Bucket composition')
plt.tight_layout(); plt.show()

## 5. Image quality & structure — SNR, entropy, edges, contours
(Template Set 4.1–4.3.) Computed by `tools/eda/image_metrics.py`.

> **SNR caveat:** we use `20·log10(mean/std)`; a *flat blurry* image has low std and
> so scores a **high** SNR. That's why SNR alone is misleading here — we combine it
> with blur/entropy/edge in the poor-quality flag.

In [ ]:
from tools.eda.image_metrics import analyze_dir, flag_poor_quality

SAMPLE_N = None   # None = all images (a few min on Colab); set e.g. 400 for a quick pass
dfq = pd.DataFrame(analyze_dir(cfg.images_dir, sample=SAMPLE_N))
dfq['poor_quality'] = flag_poor_quality(dfq)

print(dfq[['snr_db', 'entropy', 'edge_density', 'sobel_energy', 'contour_count']].describe().round(2).to_string())
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
sns.histplot(dfq['snr_db'], bins=50, ax=ax[0, 0]);        ax[0, 0].set_title('SNR (dB)')
sns.histplot(dfq['entropy'], bins=50, ax=ax[0, 1]);       ax[0, 1].set_title('Entropy (bits)')
sns.histplot(dfq['edge_density'], bins=50, ax=ax[1, 0]);  ax[1, 0].set_title('Edge density')
sns.histplot(dfq['contour_count'], bins=50, ax=ax[1, 1]); ax[1, 1].set_title('Contour count')
plt.tight_layout(); plt.show()
print(f"Poor-quality flagged: {int(dfq['poor_quality'].sum())} / {len(dfq)}")

In [ ]:
# Eyeball the extremes: flagged-poor (blurriest) vs clean (sharpest)
import cv2
def show(ids, title):
    fig, axes = plt.subplots(1, len(ids), figsize=(4 * len(ids), 4))
    for ax, bid in zip(np.atleast_1d(axes), ids):
        im = cv2.imread(str(cfg.images_dir / f'{bid}.jpg'))
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(bid); ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

show(dfq[dfq.poor_quality].sort_values('blur_var')['bin_id'].head(4).tolist(), 'Flagged POOR (blurriest)')
show(dfq[~dfq.poor_quality].sort_values('blur_var', ascending=False)['bin_id'].head(4).tolist(), 'Clean (sharpest)')

## 6. Persist enriched image metrics

In [ ]:
out = cfg.splits_dir / 'image_quality_v2.csv'
dfq.to_csv(out, index=False)
print('wrote', out, dfq.shape)
print('\nM2b complete — advanced EDA + full image-quality metrics. '
      'Feeds labeling priority (poor images) and the report/EDA deliverable.')